# 13. Open Providers — full adapter catalog

ArcLLM ships 16 adapters spanning every meaningful slice of the model
landscape: cloud commercial, cloud government, fast-inference clouds,
open-weight clouds, and self-hosted. This notebook is the **catalog
notebook** — the one you open when you need to know *which* adapter to
reach for, what each one supports, and how the four newest adapters
(Azure OpenAI, Google, Cohere, xAI) plug in.

You will learn:

- The taxonomy that organizes all 16 adapters into five categories
- How each adapter inherits the OpenAI Chat Completions baseline (and
  where the four exceptions live: Anthropic, Mistral, Azure OpenAI, Google)
- The contract for the four **new** v0.3.0 adapters — Azure OpenAI,
  Google, Cohere, xAI — with mock-driven request/response demos
- The `supports_thinking` model-metadata field, what it gates, and which
  models flip it on
- A capability matrix sourced directly from the packaged provider TOMLs
- How layered config (`~/.arc/arcllm.toml`) lets a deployment override
  any field — base URL, API key env, model catalog — without forking
  the package

This notebook is mock-only. Each adapter's HTTP boundary is patched
with canned JSON shaped like that provider's real response. Per-adapter
deep-dives (Anthropic, OpenAI) live in their own notebooks; this is the
breadth survey.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

## 1. Setup

The setup cell above does the standard Arc bootstrap. Below, we set
dummy API keys for every cloud provider so the catalog inspection cells
can construct adapters without raising `ArcLLMConfigError`. None of
these keys ever hit the network — every HTTP call in this notebook is
mocked.

In [2]:
# Set dummy keys for every cloud provider so we can construct adapters
# without hitting the api_key_required guard. No real network traffic
# happens in this notebook — every adapter call is mocked.
_DUMMY_KEY_ENV_VARS = [
    "ANTHROPIC_API_KEY",
    "OPENAI_API_KEY",
    "AZURE_OPENAI_API_KEY",
    "GOOGLE_API_KEY",
    "COHERE_API_KEY",
    "XAI_API_KEY",
    "DEEPSEEK_API_KEY",
    "GROQ_API_KEY",
    "MISTRAL_API_KEY",
    "MOONSHOT_API_KEY",
    "FIREWORKS_API_KEY",
    "TOGETHER_API_KEY",
    "HF_TOKEN",
]
for _var in _DUMMY_KEY_ENV_VARS:
    os.environ.setdefault(_var, "dummy-key-for-mock-tests")

# Confirm imports work
from arcllm import __version__, load_provider_config
from arcllm.registry import clear_cache, load_model

print(f"arcllm v{__version__} ready")
print(f"{len(_DUMMY_KEY_ENV_VARS)} dummy keys set")

arcllm v0.7.0 ready
13 dummy keys set


## 2. Adapter taxonomy

Sixteen adapters is a lot. Five categories make the choice tractable:

| Category | When to reach for it | Adapters |
|---|---|---|
| **Cloud commercial** | Frontier capability, broadest tool/vision/thinking support | `anthropic`, `openai`, `google`, `cohere`, `xai` |
| **Cloud government** | FedRAMP / IL5 / DoD environments — must use government endpoints | `azure_openai` (commercial *and* gov endpoints) |
| **Fast inference** | Low-latency loops; serving open-weight models on specialty silicon | `groq`, `fireworks`, `together` |
| **Open-source / managed** | Open-weight models with a managed control plane | `deepseek`, `mistral`, `moonshot`, `huggingface` |
| **Self-hosted** | Air-gapped, on-prem, or zero-egress deployments | `ollama`, `vllm`, `huggingface_tgi` |

The taxonomy isn't an enum or a class hierarchy — it's a mental model
for picking adapters. The code treats them all uniformly: every adapter
implements the same `LLMProvider` protocol and is loaded the same way
through `load_model()`.

In [3]:
# Map every adapter to its category at runtime, sourced from
# arcllm's lazy-import registry. This is the catalog you scan when
# choosing a provider.

ADAPTER_TAXONOMY: dict[str, list[str]] = {
    "Cloud commercial": ["anthropic", "openai", "google", "cohere", "xai"],
    "Cloud government": ["azure_openai"],
    "Fast inference": ["groq", "fireworks", "together"],
    "Open source / managed": ["deepseek", "mistral", "moonshot", "huggingface"],
    "Self-hosted": ["ollama", "vllm", "huggingface_tgi"],
}

total = sum(len(v) for v in ADAPTER_TAXONOMY.values())
print(f"Total adapters: {total}")
for category, adapters in ADAPTER_TAXONOMY.items():
    print(f"  {category:<24} {len(adapters):>2}  {', '.join(adapters)}")

Total adapters: 16
  Cloud commercial          5  anthropic, openai, google, cohere, xai
  Cloud government          1  azure_openai
  Fast inference            3  groq, fireworks, together
  Open source / managed     4  deepseek, mistral, moonshot, huggingface
  Self-hosted               3  ollama, vllm, huggingface_tgi


In [4]:
# Confirm every adapter is wired into the lazy-import registry. The
# arcllm package defers httpx imports — adapter classes are loaded on
# first attribute access — but they must all be discoverable.
import arcllm

ADAPTER_CLASS_NAMES = {
    "anthropic": "AnthropicAdapter",
    "openai": "OpenaiAdapter",
    "azure_openai": "Azure_OpenaiAdapter",
    "google": "GoogleAdapter",
    "cohere": "CohereAdapter",
    "xai": "XaiAdapter",
    "groq": "GroqAdapter",
    "fireworks": "FireworksAdapter",
    "together": "TogetherAdapter",
    "deepseek": "DeepseekAdapter",
    "mistral": "MistralAdapter",
    "moonshot": "MoonshotAdapter",
    "huggingface": "HuggingfaceAdapter",
    "huggingface_tgi": "Huggingface_TgiAdapter",
    "ollama": "OllamaAdapter",
    "vllm": "VllmAdapter",
}

missing = [n for n in ADAPTER_CLASS_NAMES.values() if not hasattr(arcllm, n)]
print(f"All {len(ADAPTER_CLASS_NAMES)} adapters discoverable: {not missing}")
print(f"Missing: {missing or 'none'}")

All 16 adapters discoverable: True
Missing: none


### Inheritance backbone

Twelve of the sixteen adapters are pure or near-pure subclasses of
`OpenaiAdapter`. The OpenAI Chat Completions API has become the de
facto open standard, and ArcLLM leans into that — most provider files
are under twenty lines.

The four genuine exceptions are:

| Adapter | Why it's not a thin alias |
|---|---|
| `anthropic` | Native Messages API: extracts `system` to a top-level field, uses `x-api-key` header, different content-block schema |
| `mistral` | OpenAI-shaped, but `tool_choice="required"` maps to `"any"`, plus an extra `model_length` finish reason |
| `azure_openai` | OpenAI-shaped, but URL is `{base}/openai/v1/chat/completions` (no `/v1` segment) and auth uses the `api-key` header instead of `Authorization: Bearer` |
| `google` | OpenAI-shaped, but URL is `{base}/chat/completions` (Google omits the `/v1/` prefix that OpenAI canonicalizes) |

Everything else (`groq`, `fireworks`, `together`, `deepseek`, `cohere`,
`xai`, `moonshot`, `ollama`, `vllm`, `huggingface`, `huggingface_tgi`)
is a name-only override on `OpenaiAdapter`.

In [5]:
import inspect

from arcllm.adapters.anthropic import AnthropicAdapter
from arcllm.adapters.azure_openai import Azure_OpenaiAdapter
from arcllm.adapters.cohere import CohereAdapter
from arcllm.adapters.google import GoogleAdapter
from arcllm.adapters.mistral import MistralAdapter
from arcllm.adapters.openai import OpenaiAdapter
from arcllm.adapters.xai import XaiAdapter

ADAPTERS = [
    AnthropicAdapter,
    OpenaiAdapter,
    Azure_OpenaiAdapter,
    GoogleAdapter,
    MistralAdapter,
    CohereAdapter,
    XaiAdapter,
]

for cls in ADAPTERS:
    parent = cls.__mro__[1].__name__
    src_lines = len(inspect.getsource(cls).splitlines())
    print(f"{cls.__name__:<22} parent={parent:<14} adapter LOC={src_lines}")

AnthropicAdapter       parent=BaseAdapter    adapter LOC=234
OpenaiAdapter          parent=BaseAdapter    adapter LOC=283
Azure_OpenaiAdapter    parent=OpenaiAdapter  adapter LOC=31
GoogleAdapter          parent=OpenaiAdapter  adapter LOC=15
MistralAdapter         parent=OpenaiAdapter  adapter LOC=31
CohereAdapter          parent=OpenaiAdapter  adapter LOC=6
XaiAdapter             parent=OpenaiAdapter  adapter LOC=6


In [6]:
# What does "thin alias" actually look like? Show the entire CohereAdapter
# source — 12 lines, name-only override.
print(inspect.getsource(CohereAdapter))

class CohereAdapter(OpenaiAdapter):
    """Thin alias for Cohere's OpenAI-compatible API."""

    @property
    def name(self) -> str:
        return "cohere"



## 3. Cloud commercial

Five adapters sit in the commercial-cloud category. Anthropic and
OpenAI are covered in their own notebooks (`03-anthropic-adapter`,
`05-openai-adapter`); here we focus on the three new arrivals plus a
quick check that the catalog still works for Anthropic / OpenAI.

| Provider | Default model | Distinguishing trait |
|---|---|---|
| Anthropic | `claude-sonnet-4-6` | Native Messages API; thinking models built-in |
| OpenAI | `gpt-4.1` | OpenAI Chat baseline; o-series flips `supports_thinking` |
| Google | `gemini-2.5-flash` | OpenAI-compatible at `generativelanguage.googleapis.com/v1beta/openai`; multimodal (text/image/audio/video) |
| Cohere | `command-a-03-2025` | OpenAI-compatible at `api.cohere.com/compatibility` |
| xAI | `grok-3` | OpenAI-compatible at `api.x.ai`; `grok-4` and `grok-3-mini` are thinking models |


In [7]:
# Show the default model and key capability flags for each commercial
# cloud provider — sourced from the packaged TOMLs, not memory.

for name in ("anthropic", "openai", "google", "cohere", "xai"):
    cfg = load_provider_config(name)
    default = cfg.provider.default_model
    meta = cfg.models.get(default) or next(iter(cfg.models.values()))
    print(
        f"{name:<10} default={default:<28} "
        f"ctx={meta.context_window:>7}  "
        f"vision={meta.supports_vision}  "
        f"thinking={meta.supports_thinking}"
    )

anthropic  default=claude-sonnet-4-6            ctx= 200000  vision=True  thinking=True
openai     default=gpt-4.1                      ctx=1000000  vision=True  thinking=False
google     default=gemini-2.5-flash             ctx=1000000  vision=True  thinking=True
cohere     default=command-a-03-2025            ctx= 128000  vision=False  thinking=False
xai        default=grok-3                       ctx= 131072  vision=False  thinking=False


## 4. Cloud government

`azure_openai` is the only adapter explicitly built for government
deployments. Azure OpenAI Service runs OpenAI's models inside
Microsoft's commercial *and* government clouds, with three quirks the
adapter handles:

1. **URL shape.** Azure's v1 API is `{base}/openai/v1/chat/completions`
   — note the `/openai` prefix and **no** `/v1/` segment between
   `openai` and `chat/completions`. Appending `?api-version=...` causes
   a hard 400 on v1.
2. **Auth header.** Azure uses the lowercase `api-key` header, not
   `Authorization: Bearer`.
3. **Endpoint host.** Same code path serves both `*.openai.azure.com`
   (commercial) and `*.openai.azure.us` (government). The host comes
   straight from `provider.base_url` — no special-casing in the
   adapter.

The model field is the **deployment name** you configured in the Azure
portal, not a canonical OpenAI model name. The packaged TOML lists
canonical names (`gpt-4_1`, `o3`, etc.) for pricing/capability
reference; production deployments override `provider.default_model` via
the user-config layer (covered in §11).

In [8]:
# Show the two endpoint flavors. The adapter doesn't care which one
# you use — the flag flips entirely on base_url.

for label, base_url in [
    ("Azure commercial", "https://my-resource.openai.azure.com"),
    ("Azure government", "https://my-resource.openai.azure.us"),
]:
    suffix = base_url.split(".azure.")[1]
    is_gov = suffix == "us"
    print(f"{label:<18} {base_url:<45}  gov={is_gov}")

Azure commercial   https://my-resource.openai.azure.com           gov=False
Azure government   https://my-resource.openai.azure.us            gov=True


## 5. Fast inference

Three adapters target the "open-weight models on specialty silicon"
slice. All three are pure OpenAI-compatible aliases — the speed comes
from the provider's hardware, not from anything in the adapter.

| Provider | Hardware | Distinguishing trait |
|---|---|---|
| Groq | Custom LPUs | Sub-second TTFT on 70B-class Llama; tool calling supported |
| Fireworks | GPUs (FireAttention) | Strong function calling, low latency on Llama 3.x |
| Together | GPUs | Cost-optimized; broad open-weight catalog including DeepSeek-V3 |

(The `cerebras` adapter doesn't exist yet — when it lands it'll be
another thin alias.)

In [9]:
# Show what's in the catalog for each fast-inference provider.
for name in ("groq", "fireworks", "together"):
    cfg = load_provider_config(name)
    print(f"\n--- {name} ({cfg.provider.base_url}) ---")
    for model_name, meta in cfg.models.items():
        print(
            f"  {model_name:<55} ctx={meta.context_window:>7}  "
            f"in/out=${meta.cost_input_per_1m}/${meta.cost_output_per_1m}"
        )


--- groq (https://api.groq.com/openai) ---
  llama-3.3-70b-versatile                                 ctx= 128000  in/out=$0.59/$0.79
  llama-3.1-8b-instant                                    ctx= 128000  in/out=$0.05/$0.08
  meta-llama/llama-4-scout-17b-16e-instruct               ctx= 512000  in/out=$0.11/$0.34
  openai/gpt-oss-120b                                     ctx= 128000  in/out=$0.0/$0.0
  qwen/qwen3-32b                                          ctx= 128000  in/out=$0.0/$0.0

--- fireworks (https://api.fireworks.ai/inference) ---
  accounts/fireworks/models/llama-v3p3-70b-instruct       ctx= 128000  in/out=$0.9/$0.9
  accounts/fireworks/models/llama-v3p1-8b-instruct        ctx= 128000  in/out=$0.2/$0.2
  accounts/fireworks/models/deepseek-v3                   ctx= 128000  in/out=$0.9/$0.9
  accounts/fireworks/models/qwen2p5-72b-instruct          ctx=  32768  in/out=$0.9/$0.9

--- together (https://api.together.xyz) ---
  meta-llama/Llama-3.3-70B-Instruct-Turbo                

## 6. Open source / managed

Four adapters give you open-weight models behind a managed control
plane (no GPUs to run, no kubectl).

- `deepseek` — DeepSeek's own API. `deepseek-chat` and
  `deepseek-reasoner` (the latter flips `supports_thinking=true`).
- `mistral` — Mistral's API. The only OpenAI-shaped adapter that
  needs *behavior* overrides, not just URL/auth — see §8.
- `moonshot` — Moonshot AI's Kimi line. `kimi-k2.5` adds vision.
- `huggingface` — HuggingFace's hosted Inference API. Default key
  env is `HF_TOKEN`. Models are referenced by hub path
  (`meta-llama/Llama-3.3-70B-Instruct`).

In [10]:
# Open-source / managed providers and their default models
for name in ("deepseek", "mistral", "moonshot", "huggingface"):
    cfg = load_provider_config(name)
    print(
        f"{name:<14} default={cfg.provider.default_model:<40} key_env={cfg.provider.api_key_env}"
    )

deepseek       default=deepseek-chat                            key_env=DEEPSEEK_API_KEY
mistral        default=mistral-large-latest                     key_env=MISTRAL_API_KEY
moonshot       default=kimi-k2.5                                key_env=MOONSHOT_API_KEY
huggingface    default=meta-llama/Llama-3.3-70B-Instruct        key_env=HF_TOKEN


## 7. Self-hosted

Three adapters point at servers *you* run. The defining property is
`api_key_required = false` — local servers don't need bearer tokens —
plus an HTTP (not HTTPS) `base_url`. ArcLLM only allows plain HTTP
when the host is `localhost`, `127.0.0.1`, or `[::1]`; remote
self-hosted endpoints must use HTTPS.

| Provider | Default URL | Use case |
|---|---|---|
| `ollama` | `http://localhost:11434` | Dev box, single-machine, ggml-quant models |
| `vllm` | `http://localhost:8000` | High-throughput serving on a GPU box |
| `huggingface_tgi` | `http://localhost:8080` | Self-hosted Text Generation Inference |

All three set every cost field to zero — there's no API meter when the
GPU's in your closet. Telemetry still records token counts.

In [11]:
# Self-hosted providers — confirm api_key_required=False and zero cost.
for name in ("ollama", "vllm", "huggingface_tgi"):
    cfg = load_provider_config(name)
    default = cfg.provider.default_model
    meta = cfg.models.get(default) or next(iter(cfg.models.values()))
    print(
        f"{name:<18} url={cfg.provider.base_url:<28} "
        f"api_key_required={cfg.provider.api_key_required}  "
        f"cost_in/out=${meta.cost_input_per_1m}/${meta.cost_output_per_1m}"
    )

ollama             url=http://localhost:11434       api_key_required=False  cost_in/out=$0.0/$0.0
vllm               url=http://localhost:8000        api_key_required=False  cost_in/out=$0.0/$0.0
huggingface_tgi    url=http://localhost:8080        api_key_required=False  cost_in/out=$0.0/$0.0


In [12]:
# The HTTPS guard — try a remote http:// URL and watch validation reject it.
from pydantic import ValidationError

from arcllm.config import ProviderSettings

try:
    ProviderSettings(
        api_format="openai-chat",
        base_url="http://my-llm-host.example.com",
        api_key_env="X",
        api_key_required=False,
        default_model="m",
        default_temperature=0.7,
    )
    print("UNEXPECTED: validation passed")
except ValidationError as exc:
    # First error message is enough to confirm the guard fired.
    print(f"Validation rejected http:// for remote host: {exc.errors()[0]['type']}")

Validation rejected http:// for remote host: value_error


## 8. The four new v0.3.0 adapters — mock-driven demos

Azure OpenAI, Google, Cohere, and xAI all landed in v0.3.0. The next
four cells each patch one adapter's HTTP boundary with a canned
response shaped like that provider's actual API, then invoke it and
inspect the unified `LLMResponse`.

This is the same mocking pattern used in
`packages/arcllm/tests/test_azure_openai.py` and friends — we replace
`httpx.AsyncClient.post` with an `AsyncMock` returning a fake
`httpx.Response`, build the adapter manually with an in-memory
`ProviderConfig`, and call `await adapter.invoke(...)`.

In [13]:
# Shared helpers for mocking httpx and building adapters in-memory.
# We use top-level `await` directly in cells (Jupyter supports it) instead
# of `asyncio.run()`, which would conflict with the kernel's running loop.
from unittest.mock import AsyncMock, MagicMock

import httpx

from arcllm.config import ModelMetadata, ProviderConfig, ProviderSettings
from arcllm.types import LLMResponse, Message


def make_config(
    *,
    base_url: str,
    api_key_env: str,
    default_model: str,
    api_key_required: bool = True,
    supports_thinking: bool = False,
) -> ProviderConfig:
    """Build a minimal ProviderConfig for the mock demos."""
    return ProviderConfig(
        provider=ProviderSettings(
            api_format="openai-chat",
            base_url=base_url,
            api_key_env=api_key_env,
            api_key_required=api_key_required,
            default_model=default_model,
            default_temperature=0.7,
        ),
        models={
            default_model: ModelMetadata(
                context_window=128000,
                max_output_tokens=4096,
                supports_tools=True,
                supports_vision=False,
                supports_thinking=supports_thinking,
                input_modalities=["text"],
                cost_input_per_1m=1.0,
                cost_output_per_1m=2.0,
                cost_cache_read_per_1m=0.0,
                cost_cache_write_per_1m=0.0,
            )
        },
    )


def fake_chat_completion(
    content: str = "Hello from a mocked provider.",
    finish_reason: str = "stop",
    model: str = "test-model",
) -> dict:
    """Build a generic OpenAI-Chat-shaped response body."""
    return {
        "id": "chatcmpl-mock",
        "object": "chat.completion",
        "model": model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": content},
                "finish_reason": finish_reason,
            }
        ],
        "usage": {
            "prompt_tokens": 12,
            "completion_tokens": 8,
            "total_tokens": 20,
        },
    }


def make_mock_post(response_json: dict, status_code: int = 200) -> AsyncMock:
    """Build an AsyncMock returning a fake httpx.Response carrying JSON."""
    fake_response = MagicMock(spec=httpx.Response)
    fake_response.status_code = status_code
    fake_response.json.return_value = response_json
    fake_response.text = str(response_json)
    fake_response.headers = {}
    return AsyncMock(return_value=fake_response)


print("Mock helpers loaded.")

Mock helpers loaded.


### 8.1 Azure OpenAI — mock demo

Watch for two things in the URL the adapter posts to:

1. The path is `/openai/v1/chat/completions` (no `/v1/` between
   `openai` and `chat/completions`).
2. The auth header is `api-key`, not `Authorization`.

We'll capture the call args from the mock to prove both.

In [14]:
from arcllm.adapters.azure_openai import Azure_OpenaiAdapter

azure_cfg = make_config(
    base_url="https://my-resource.openai.azure.us",  # gov endpoint
    api_key_env="AZURE_OPENAI_API_KEY",
    default_model="my-gpt4o-deployment",
)

azure_adapter = Azure_OpenaiAdapter(azure_cfg, "my-gpt4o-deployment")
mock_post = make_mock_post(
    fake_chat_completion(
        content="Azure says hi.",
        model="my-gpt4o-deployment",
    )
)
azure_adapter._client.post = mock_post  # patch the boundary

result: LLMResponse = await azure_adapter.invoke([Message(role="user", content="Hello?")])

call_kwargs = mock_post.call_args.kwargs
posted_url = mock_post.call_args.args[0]
print(f"URL posted: {posted_url}")
print(f"Auth header key: {[k for k in call_kwargs['headers'] if 'key' in k.lower()]}")
print(f"Response content: {result.content!r}")
print(f"Stop reason: {result.stop_reason}")
print(f"Tokens: in={result.usage.input_tokens} out={result.usage.output_tokens}")

URL posted: https://my-resource.openai.azure.us/openai/v1/chat/completions
Auth header key: ['api-key']
Response content: 'Azure says hi.'
Stop reason: end_turn
Tokens: in=12 out=8


In [15]:
# Azure's distinctive finish_reason: 'content_filter' from Azure
# Content Safety. The base OpenAI adapter already maps it.
mock_post = make_mock_post(
    fake_chat_completion(
        content=None,
        finish_reason="content_filter",
        model="my-gpt4o-deployment",
    )
)
azure_adapter._client.post = mock_post
result = await azure_adapter.invoke([Message(role="user", content="...")])
print(f"Stop reason from Azure content_filter: {result.stop_reason}")

Stop reason from Azure content_filter: content_filter


### 8.2 Google Gemini — mock demo

Google publishes an OpenAI-compatible endpoint at
`generativelanguage.googleapis.com/v1beta/openai`. The URL quirk is
that it's `{base}/chat/completions` — no `/v1/` separator, because the
`/v1beta` is already in the base URL.

`gemini-2.5-flash` and `gemini-2.5-pro` both flip `supports_thinking`,
which means OpenAI-style `system` → `developer` role mapping kicks in.
We'll verify the URL and the role mapping in the demo.

In [16]:
from arcllm.adapters.google import GoogleAdapter

google_cfg = make_config(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",
    api_key_env="GOOGLE_API_KEY",
    default_model="gemini-2.5-flash",
    supports_thinking=True,  # 2.5-flash is a thinking model
)
google_adapter = GoogleAdapter(google_cfg, "gemini-2.5-flash")
mock_post = make_mock_post(
    fake_chat_completion(content="Hi from Gemini.", model="gemini-2.5-flash")
)
google_adapter._client.post = mock_post

result = await google_adapter.invoke(
    [
        Message(role="system", content="You are precise."),
        Message(role="user", content="Hello?"),
    ]
)

posted_url = mock_post.call_args.args[0]
posted_body = mock_post.call_args.kwargs["json"]
print(f"URL posted: {posted_url}")
print(f"First message role (was 'system'): {posted_body['messages'][0]['role']}")
print(f"Response: {result.content!r}")

URL posted: https://generativelanguage.googleapis.com/v1beta/openai/chat/completions
First message role (was 'system'): developer
Response: 'Hi from Gemini.'


### 8.3 Cohere — mock demo

Cohere's OpenAI-compatible endpoint is at
`api.cohere.com/compatibility`. The adapter is a name-only override —
no URL or auth quirks — so this is mostly a sanity demo confirming the
inherited OpenAI request/response cycle works against a Cohere-shaped
TOML.

In [17]:
from arcllm.adapters.cohere import CohereAdapter

cohere_cfg = make_config(
    base_url="https://api.cohere.com/compatibility",
    api_key_env="COHERE_API_KEY",
    default_model="command-a-03-2025",
)
cohere_adapter = CohereAdapter(cohere_cfg, "command-a-03-2025")
mock_post = make_mock_post(
    fake_chat_completion(content="Cohere reply.", model="command-a-03-2025")
)
cohere_adapter._client.post = mock_post

result = await cohere_adapter.invoke([Message(role="user", content="Ping?")])
posted_url = mock_post.call_args.args[0]
print(f"URL posted: {posted_url}")  # expect /v1/chat/completions suffix
print(f"Adapter name: {cohere_adapter.name}")
print(f"Response: {result.content!r} (stop={result.stop_reason})")

URL posted: https://api.cohere.com/compatibility/v1/chat/completions
Adapter name: cohere
Response: 'Cohere reply.' (stop=end_turn)


### 8.4 xAI Grok — mock demo

`xai` is also a thin alias. The interesting bit is the model catalog:
`grok-4` and `grok-3-mini` both set `supports_thinking=true`, and
`grok-2-vision-1212` sets `supports_vision=true` but `supports_tools=false`.
The adapter doesn't care — it inherits `_is_reasoning_model` from
OpenaiAdapter, which keys off the model metadata.

In [18]:
from arcllm.adapters.xai import XaiAdapter

xai_cfg = make_config(
    base_url="https://api.x.ai",
    api_key_env="XAI_API_KEY",
    default_model="grok-3-mini",
    supports_thinking=True,
)
xai_adapter = XaiAdapter(xai_cfg, "grok-3-mini")
mock_post = make_mock_post(
    fake_chat_completion(content="Grok reasoning here.", model="grok-3-mini")
)
xai_adapter._client.post = mock_post

result = await xai_adapter.invoke(
    [
        Message(role="system", content="Think carefully."),
        Message(role="user", content="What's 2+2?"),
    ]
)
posted_body = mock_post.call_args.kwargs["json"]
print(f"Adapter name: {xai_adapter.name}")
print(f"Posted model: {posted_body['model']}")
print(f"system→developer applied: {posted_body['messages'][0]['role'] == 'developer'}")
print(f"max_completion_tokens used (reasoning model): {'max_completion_tokens' in posted_body}")
print(f"Response: {result.content!r}")

Adapter name: xai
Posted model: grok-3-mini
system→developer applied: True
max_completion_tokens used (reasoning model): True
Response: 'Grok reasoning here.'


In [19]:
# Mistral's tool_choice quirk — included here because it's the one
# OpenAI-shaped adapter that overrides _build_request_body, not just
# the URL or auth.
from arcllm.adapters.mistral import _MISTRAL_STOP_REASON_MAP, MistralAdapter
from arcllm.types import Tool

mistral_cfg = make_config(
    base_url="https://api.mistral.ai",
    api_key_env="MISTRAL_API_KEY",
    default_model="mistral-large-latest",
)
mistral_adapter = MistralAdapter(mistral_cfg, "mistral-large-latest")

body = mistral_adapter._build_request_body(
    messages=[Message(role="user", content="hi")],
    tools=[Tool(name="calc", description="x", parameters={"type": "object"})],
    tool_choice="required",  # OpenAI keyword
)
print(f"OpenAI 'required' translated to: {body['tool_choice']!r}  (Mistral expects 'any')")
print(f"Mistral stop_reason map: {_MISTRAL_STOP_REASON_MAP}")

OpenAI 'required' translated to: 'any'  (Mistral expects 'any')
Mistral stop_reason map: {'stop': 'end_turn', 'tool_calls': 'tool_use', 'length': 'max_tokens', 'model_length': 'max_tokens', 'content_filter': 'content_filter'}


## 9. The `supports_thinking` metadata field

`ModelMetadata.supports_thinking` is a per-model boolean in the
provider TOML (`v0.4.0`). It marks the model as a *reasoning model*
— Anthropic extended-thinking, OpenAI o-series, DeepSeek-Reasoner,
Grok thinking variants, and Gemini 2.5 family.

Three things change when the flag is true:

1. **Role mapping.** `OpenaiAdapter._format_messages()` rewrites
   `role="system"` to `role="developer"` for the affected models.
   OpenAI's o-series and Azure's o-series deployments reject
   `system`. Anthropic's adapter never sees this — it has its own
   path.
2. **Body shape.** `_build_request_body()` swaps `max_tokens` for
   `max_completion_tokens` and adds `reasoning_effort`
   (`low`/`medium`/`high`, default `medium`). Reasoning models
   currently reject the older keys.
3. **Usage parsing.** `Usage.reasoning_tokens` is populated when the
   provider returns `completion_tokens_details.reasoning_tokens`.

In [20]:
# Inventory every model across every adapter where supports_thinking=True.
# This is the canonical list of "reasoning models" in the v0.4.0 catalogs.

thinking_models: list[tuple[str, str]] = []
for provider in (
    ADAPTER_TAXONOMY["Cloud commercial"]
    + ADAPTER_TAXONOMY["Cloud government"]
    + ADAPTER_TAXONOMY["Fast inference"]
    + ADAPTER_TAXONOMY["Open source / managed"]
    + ADAPTER_TAXONOMY["Self-hosted"]
):
    cfg = load_provider_config(provider)
    for model_name, meta in cfg.models.items():
        if meta.supports_thinking:
            thinking_models.append((provider, model_name))

print(f"Reasoning models in catalog: {len(thinking_models)}")
for prov, model in thinking_models:
    print(f"  {prov:<14} {model}")

Reasoning models in catalog: 22
  anthropic      claude-opus-4-6
  anthropic      claude-sonnet-4-6
  anthropic      claude-opus-4-5-20251101
  anthropic      claude-sonnet-4-5-20250929
  anthropic      claude-sonnet-4-20250514
  anthropic      claude-opus-4-20250514
  openai         o3
  openai         o3-mini
  openai         o4-mini
  google         gemini-2.5-pro
  google         gemini-2.5-flash
  xai            grok-4
  xai            grok-3-mini
  azure_openai   o1
  azure_openai   o3
  azure_openai   o3-mini
  azure_openai   o4-mini
  deepseek       deepseek-reasoner
  mistral        magistral-medium-latest
  mistral        magistral-small-latest
  ollama         deepseek-r1:8b
  ollama         gemma3


In [21]:
# Demonstrate body shape swap when supports_thinking=True. We reuse the
# xAI adapter (already configured with a thinking model in §8.4) and a
# matched non-thinking adapter to show the diff.
from arcllm.adapters.openai import OpenaiAdapter

# Thinking branch: xAI grok-3-mini (supports_thinking=True).
thinking_body = xai_adapter._build_request_body(
    messages=[Message(role="user", content="reason about this")],
)

# Non-thinking branch: build a vanilla OpenAI adapter on a plain model.
plain_cfg = make_config(
    base_url="https://api.openai.com",
    api_key_env="OPENAI_API_KEY",
    default_model="gpt-4.1-mini",
    supports_thinking=False,
)
plain_adapter = OpenaiAdapter(plain_cfg, "gpt-4.1-mini")
plain_body = plain_adapter._build_request_body(
    messages=[Message(role="user", content="just answer")],
)

print("Thinking model body keys:", sorted(thinking_body))
print("Plain   model body keys:", sorted(plain_body))
print()
print(f"Thinking has max_completion_tokens: {'max_completion_tokens' in thinking_body}")
print(f"Plain    has max_tokens          : {'max_tokens' in plain_body}")
print(f"Thinking has reasoning_effort    : {'reasoning_effort' in thinking_body}")

Thinking model body keys: ['max_completion_tokens', 'messages', 'model', 'reasoning_effort']
Plain   model body keys: ['max_tokens', 'messages', 'model', 'temperature']

Thinking has max_completion_tokens: True
Plain    has max_tokens          : True
Thinking has reasoning_effort    : True


In [22]:
# Confirm the system→developer role mapping fires only when
# supports_thinking=True, by formatting the same messages through both
# adapters.

messages = [
    Message(role="system", content="You are concise."),
    Message(role="user", content="Hi"),
]

formatted_thinking = xai_adapter._format_messages(messages)
formatted_plain = plain_adapter._format_messages(messages)

print("Thinking adapter (xai/grok-3-mini):")
for m in formatted_thinking:
    print(f"  role={m['role']:<10} content={m.get('content', '')!r}")

print("\nPlain adapter (openai/gpt-4.1-mini):")
for m in formatted_plain:
    print(f"  role={m['role']:<10} content={m.get('content', '')!r}")

Thinking adapter (xai/grok-3-mini):
  role=developer  content='You are concise.'
  role=user       content='Hi'

Plain adapter (openai/gpt-4.1-mini):
  role=system     content='You are concise.'
  role=user       content='Hi'


## 10. Capability matrix

A single table comparing all 16 adapters across the dimensions that
actually drive provider choice. Every cell is sourced from the
packaged TOML — there are no hand-edited fields below.

Columns:

- **default**: `provider.default_model`
- **ctx (k)**: default model's `context_window`, divided by 1000
- **tools / vision / thinking**: capability flags on the default model
- **modalities**: `input_modalities` (`t`=text, `i`=image, `a`=audio, `v`=video)
- **key req?**: `provider.api_key_required`
- **finish_reason map**: provider-specific stop reasons (where they differ from the OpenAI baseline)

In [23]:
# Build the matrix programmatically. One row per adapter; every value
# pulled from the TOML — no guesses, no hand-editing.
from itertools import chain

ALL_PROVIDERS = list(chain.from_iterable(ADAPTER_TAXONOMY.values()))


def modality_short(mods: list[str]) -> str:
    return "".join(
        {"text": "t", "image": "i", "audio": "a", "video": "v"}.get(m, "?") for m in mods
    )


# Adapter-level finish_reason notes, sourced from adapter source.
EXTRA_FINISH_REASONS: dict[str, str] = {
    "anthropic": "end_turn/tool_use/max_tokens/stop_sequence (native)",
    "openai": "stop/tool_calls/length/content_filter (baseline)",
    "azure_openai": "+content_filter (Azure Content Safety)",
    "mistral": "+model_length (-> max_tokens)",
}

print(
    f"{'provider':<16} {'default':<38} {'ctx':>5} {'tool':<5} {'vis':<4} "
    f"{'think':<6} {'mod':<6} {'keyreq':<7} {'finish_reason notes'}"
)
print("-" * 130)
for provider in ALL_PROVIDERS:
    cfg = load_provider_config(provider)
    default = cfg.provider.default_model
    # The default may not be a key in cfg.models — Azure's catalog uses
    # canonical names like `gpt-4_1` for models, but its `default_model`
    # is `gpt-4-1` (deployment-name placeholder). Fall back to the first
    # listed model when default isn't catalogued.
    meta = cfg.models.get(default) or next(iter(cfg.models.values()))
    shown = default if default in cfg.models else f"{default} (->{next(iter(cfg.models))})"
    note = EXTRA_FINISH_REASONS.get(provider, "(inherits OpenAI map)")
    print(
        f"{provider:<16} {shown[:36]:<38} "
        f"{meta.context_window // 1000:>4}k "
        f"{meta.supports_tools!s:<5} "
        f"{meta.supports_vision!s:<4} "
        f"{meta.supports_thinking!s:<6} "
        f"{modality_short(meta.input_modalities):<6} "
        f"{cfg.provider.api_key_required!s:<7} "
        f"{note}"
    )

provider         default                                  ctx tool  vis  think  mod    keyreq  finish_reason notes
----------------------------------------------------------------------------------------------------------------------------------
anthropic        claude-sonnet-4-6                       200k True  True True   ti     True    end_turn/tool_use/max_tokens/stop_sequence (native)
openai           gpt-4.1                                1000k True  True False  ti     True    stop/tool_calls/length/content_filter (baseline)
google           gemini-2.5-flash                       1000k True  True True   tiav   True    (inherits OpenAI map)
cohere           command-a-03-2025                       128k True  False False  t      True    (inherits OpenAI map)
xai              grok-3                                  131k True  False False  t      True    (inherits OpenAI map)
azure_openai     gpt-4.1                                 300k True  True False  ti     True    +content_filter

In [24]:
# Roll up the matrix into a few headline counts.
counts = {"tools": 0, "vision": 0, "thinking": 0, "audio": 0, "video": 0}
for provider in ALL_PROVIDERS:
    cfg = load_provider_config(provider)
    meta = cfg.models.get(cfg.provider.default_model) or next(iter(cfg.models.values()))
    counts["tools"] += int(meta.supports_tools)
    counts["vision"] += int(meta.supports_vision)
    counts["thinking"] += int(meta.supports_thinking)
    counts["audio"] += int("audio" in meta.input_modalities)
    counts["video"] += int("video" in meta.input_modalities)

print("Adapters where the *default* model supports each capability:")
for cap, n in counts.items():
    print(f"  {cap:<10} {n:>2} / {len(ALL_PROVIDERS)}")

Adapters where the *default* model supports each capability:
  tools      16 / 16
  vision      6 / 16
  thinking    2 / 16
  audio       1 / 16
  video       1 / 16


In [25]:
# load_model() smoke-test across every adapter — no network calls; we
# tear the adapter down before any HTTP would happen.
results: list[tuple[str, str]] = []
for provider in ALL_PROVIDERS:
    clear_cache()
    model = load_model(provider)
    # The chain is module-wrapped; .name comes from the innermost adapter.
    inner = model
    while hasattr(inner, "_inner"):
        inner = inner._inner
    results.append((provider, inner.name))
    await model.close()

for provider, name in results:
    match = "OK" if name == provider else "(adapter.name=" + name + ")"
    print(f"  {provider:<16} -> {match}")
print(f"\n{len(results)} / 16 adapters loaded")

  anthropic        -> OK
  openai           -> OK
  google           -> OK
  cohere           -> OK
  xai              -> OK
  azure_openai     -> OK
  groq             -> OK
  fireworks        -> OK
  together         -> OK
  deepseek         -> OK
  mistral          -> OK
  moonshot         -> OK
  huggingface      -> OK
  ollama           -> OK
  vllm             -> OK
  huggingface_tgi  -> OK

16 / 16 adapters loaded


## 11. Layered config

The packaged provider TOMLs at
`packages/arcllm/src/arcllm/providers/*.toml` are the **base layer**.
A user-config file at `${ARC_CONFIG_DIR:-~/.arc}/arcllm.toml`
deep-merges over the packaged `config.toml` (the *global* config, not
the per-provider files) — this is how you override defaults like
provider, temperature, module toggles, or vault config without forking
the package.

For the deep dive on layered config and deep-merge semantics, see
`arcllm/02-config-loading.ipynb`. Here we just demonstrate the two
patterns most relevant to a multi-adapter deployment:

1. **Override the default provider** — flip a global setting so
   `load_model()` (no provider argument) hits Azure instead of
   Anthropic.
2. **Override module defaults** — turn audit logging on for every
   model that's loaded, without touching code.

Per-provider TOMLs are not currently overlaid by the user-config
layer. Customizing a provider's `base_url` or model catalog requires
either an environment variable substitution at deployment time (the
`base_url` is a plain string), or — until per-provider overlay lands
— forking the TOML.

In [26]:
# Demonstrate the global-config overlay. We won't ship a real
# ~/.arc/arcllm.toml — instead, point ARC_CONFIG_DIR at a tempdir and
# write a file there.
import tempfile
import textwrap

from arcllm.config import load_global_config

with tempfile.TemporaryDirectory() as tmp:
    user_overlay_dir = Path(tmp)
    overlay_text = textwrap.dedent(
        """
        [defaults]
        provider = "azure_openai"
        temperature = 0.2

        [modules.audit]
        enabled = true
        """
    ).strip()
    (user_overlay_dir / "arcllm.toml").write_text(overlay_text)

    os.environ["ARC_CONFIG_DIR"] = str(user_overlay_dir)
    cfg = load_global_config()
    print(f"defaults.provider    = {cfg.defaults.provider}")
    print(f"defaults.temperature = {cfg.defaults.temperature}")
    print(f"modules.audit.enabled = {cfg.modules['audit'].enabled}")

# Clean up so the env var doesn't leak into later cells.
os.environ.pop("ARC_CONFIG_DIR", None)

defaults.provider    = azure_openai
defaults.temperature = 0.2
modules.audit.enabled = True


'/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/tmpujc456jd'

In [27]:
# Without a user overlay, packaged defaults stand alone. Confirm that
# the packaged config.toml has plausible values.
cfg = load_global_config()
print("Packaged defaults:")
print(f"  provider    = {cfg.defaults.provider}")
print(f"  temperature = {cfg.defaults.temperature}")
print(f"  max_tokens  = {cfg.defaults.max_tokens}")
print(f"  modules     = {sorted(cfg.modules)}")

Packaged defaults:
  provider    = anthropic
  temperature = 0.7
  max_tokens  = 4096
  modules     = ['audit', 'circuit_breaker', 'fallback', 'guardrails', 'injection', 'load_balance', 'otel', 'queue', 'rate_limit', 'retry', 'routing', 'security', 'telemetry']


In [28]:
# Provider name validation (ASI-04 hardening, v0.4.0). Tries to block
# path-traversal-style attacks on load_provider_config().
from arcllm.exceptions import ArcLLMConfigError

bad_names = ["../etc/passwd", "azure-openai", "Azure_OpenAI", "", "a" * 65]
for name in bad_names:
    try:
        load_provider_config(name)
        print(f"  {name!r:<24} UNEXPECTED PASS")
    except ArcLLMConfigError as exc:
        msg = str(exc).split(".")[0]
        print(f"  {name!r:<24} rejected: {msg}")

  '../etc/passwd'          rejected: Invalid provider name '
  'azure-openai'           rejected: Invalid provider name 'azure-openai'
  'Azure_OpenAI'           rejected: Invalid provider name 'Azure_OpenAI'
  ''                       rejected: Provider name cannot be empty
  'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa' rejected: Provider name too long (max 64 characters)


## 12. Summary

Sixteen adapters, organized by intended deployment surface:

- **Cloud commercial (5)** — `anthropic`, `openai`, `google`,
  `cohere`, `xai`. Frontier capability; native or
  OpenAI-compatible APIs.
- **Cloud government (1)** — `azure_openai` covers both
  `*.azure.com` and `*.azure.us` from one adapter; quirks are
  URL-shape and `api-key` header.
- **Fast inference (3)** — `groq`, `fireworks`, `together`. Pure
  thin aliases; speed comes from hardware.
- **Open-source / managed (4)** — `deepseek`, `mistral`,
  `moonshot`, `huggingface`. `mistral` is the only one with
  behavior overrides (`tool_choice` and `model_length`).
- **Self-hosted (3)** — `ollama`, `vllm`, `huggingface_tgi`.
  `api_key_required=False`, HTTP-localhost permitted.

The new v0.3.0 adapters — Azure OpenAI, Google, Cohere, xAI — were
each demonstrated end-to-end with mocked HTTP, confirming URL,
auth, and response shape.

`supports_thinking` is the v0.4.0 metadata field that marks
reasoning models. When true, the OpenAI-shaped adapters swap
`max_tokens`→`max_completion_tokens`, add `reasoning_effort`,
remap `system`→`developer`, and parse `reasoning_tokens` from
usage details.

The capability matrix (§10) is sourced entirely from packaged
TOMLs — no hand-edits — so refreshes to the catalog stay in
sync with the notebook automatically.

Layered config (§11) overlays user settings from
`${ARC_CONFIG_DIR:-~/.arc}/arcllm.toml` over packaged defaults,
deep-merging dicts and replacing scalars; per-provider overlay is
not yet supported. See `arcllm/02-config-loading` for the deep
dive.

**Next:** `arcllm/03-anthropic-adapter` and
`arcllm/05-openai-adapter` for per-adapter deep-dives;
`arcllm/15-queue-circuit-breaker` and
`arcllm/16-config-controller` for the resilience and
runtime-config primitives that compose with these adapters.

**Public API exercised:**

- `arcllm.load_provider_config`, `arcllm.load_global_config`
- `arcllm.load_model`, `arcllm.clear_cache`
- `arcllm.ModelMetadata`, `arcllm.ProviderConfig`,
  `arcllm.ProviderSettings`, `arcllm.GlobalConfig`,
  `arcllm.DefaultsConfig`, `arcllm.ModuleConfig`
- `arcllm.types.Message`, `arcllm.types.Tool`,
  `arcllm.types.LLMResponse`
- All 16 adapter classes:
  `AnthropicAdapter`, `OpenaiAdapter`, `Azure_OpenaiAdapter`,
  `GoogleAdapter`, `CohereAdapter`, `XaiAdapter`,
  `GroqAdapter`, `FireworksAdapter`, `TogetherAdapter`,
  `DeepseekAdapter`, `MistralAdapter`, `MoonshotAdapter`,
  `HuggingfaceAdapter`, `Huggingface_TgiAdapter`,
  `OllamaAdapter`, `VllmAdapter`
- `arcllm.exceptions.ArcLLMConfigError`
